# CAVA Group (CAVA) - Data Collection
## Consumer Sector Case Study

This notebook collects data from three sources:
1. **Financial data** - Stock price + quarterly fundamentals via yfinance
2. **Google Trends** - Consumer interest proxy via pytrends
3. **Reddit sentiment** - Social media consumer sentiment via PRAW

All data is saved to CSV for use in the modeling notebook.

In [ ]:
# Install dependencies (run once)
# !pip install yfinance pytrends praw pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print('Libraries loaded.')

---
## Part 1: Financial Data (yfinance)

In [ ]:
import yfinance as yf

# --- 1A: Stock Price History ---
ticker = yf.Ticker('CAVA')

# CAVA IPO'd June 2023, so start from there
price_df = ticker.history(start='2023-06-01', end='2025-04-28')
price_df = price_df[['Close', 'Volume']].reset_index()
price_df.columns = ['date', 'close_price', 'volume']
price_df['date'] = pd.to_datetime(price_df['date']).dt.tz_localize(None)

print(f'Price data: {len(price_df)} rows, {price_df.date.min().date()} to {price_df.date.max().date()}')
price_df.head()

In [ ]:
# --- 1B: Manually entered quarterly fundamentals ---
# Source: CAVA earnings releases & 10-Q filings (SEC Edgar / IR site)
# Key metrics: Same-Restaurant Sales Growth (SSSG), AUV, Restaurant Count, Restaurant-Level Profit Margin

# Data sourced from CAVA earnings releases:
# https://ir.cava.com/news-releases

quarterly_data = {
    'quarter': [
        'Q2 2022', 'Q3 2022', 'Q4 2022',
        'Q1 2023', 'Q2 2023', 'Q3 2023', 'Q4 2023',
        'Q1 2024', 'Q2 2024', 'Q3 2024', 'Q4 2024'
    ],
    'period_end': [
        '2022-07-10', '2022-10-02', '2023-01-01',
        '2023-04-16', '2023-07-09', '2023-10-01', '2023-12-31',
        '2024-04-14', '2024-07-14', '2024-10-06', '2024-12-29'
    ],
    # Same-restaurant sales growth (%) - YoY
    'sssg_pct': [
        10.5, 12.8, 8.2,
        9.4, 18.0, 14.4, 11.6,
        2.3, 14.4, 18.1, 21.2
    ],
    # Total revenue ($M)
    'revenue_m': [
        133.1, 152.5, 148.3,
        152.8, 172.9, 187.2, 190.8,
        256.0, 231.6, 244.3, 255.5  # Note: Q1 2024 includes 53rd week effect
    ],
    # Number of restaurants (end of period)
    'restaurant_count': [
        188, 196, 208,
        219, 228, 244, 263,
        309, 323, 341, 367
    ],
    # Restaurant-level profit margin (%)
    'restaurant_margin_pct': [
        20.8, 22.1, 19.4,
        20.8, 24.0, 25.1, 22.8,
        24.9, 26.5, 26.3, 25.0
    ],
    # Average Unit Volume annualized ($M) - estimated from revenue/count
    'auv_annualized_m': [
        2.60, 2.72, 2.63,
        2.58, 2.80, 2.85, 2.83,
        2.91, 2.95, 3.01, 3.10
    ]
}

q_df = pd.DataFrame(quarterly_data)
q_df['period_end'] = pd.to_datetime(q_df['period_end'])
q_df = q_df.sort_values('period_end').reset_index(drop=True)

print('Quarterly fundamentals loaded:')
q_df

In [ ]:
# --- 1C: Visualize core KPIs ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('CAVA Group – Core KPI Trends', fontsize=15, fontweight='bold', y=1.01)

# SSSG
ax = axes[0, 0]
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in q_df['sssg_pct']]
ax.bar(q_df['quarter'], q_df['sssg_pct'], color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Same-Restaurant Sales Growth (%)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel('%')

# Revenue
ax = axes[0, 1]
ax.plot(q_df['quarter'], q_df['revenue_m'], marker='o', color='#3498db', linewidth=2)
ax.set_title('Quarterly Revenue ($M)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel('$M')

# Restaurant Count
ax = axes[1, 0]
ax.plot(q_df['quarter'], q_df['restaurant_count'], marker='s', color='#9b59b6', linewidth=2)
ax.set_title('Restaurant Count (End of Period)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel('Count')

# Restaurant Margin
ax = axes[1, 1]
ax.plot(q_df['quarter'], q_df['restaurant_margin_pct'], marker='^', color='#e67e22', linewidth=2)
ax.set_title('Restaurant-Level Profit Margin (%)', fontweight='bold')
ax.tick_params(axis='x', rotation=45)
ax.set_ylabel('%')
ax.set_ylim(15, 30)

plt.tight_layout()
plt.savefig('cava_kpis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cava_kpis.png')

---
## Part 2: Google Trends (pytrends)

We use Google Trends as a high-frequency (weekly) proxy for consumer interest in CAVA.
Hypothesis: Rising search interest leads same-restaurant sales growth by 1-2 quarters.

In [ ]:
from pytrends.request import TrendReq
import time

pytrends = TrendReq(hl='en-US', tz=360)

# Search terms - CAVA vs fast casual competitors
keywords = ['CAVA restaurant', 'Chipotle', 'Sweetgreen']

pytrends.build_payload(
    keywords,
    cat=0,
    timeframe='2022-01-01 2025-04-01',
    geo='US'
)

time.sleep(1)  # Be polite to the API
trends_df = pytrends.interest_over_time()

if 'isPartial' in trends_df.columns:
    trends_df = trends_df.drop(columns=['isPartial'])

trends_df = trends_df.reset_index()
trends_df.columns = ['date'] + [k.replace(' ', '_').lower() for k in keywords]

print(f'Google Trends data: {len(trends_df)} weeks')
trends_df.tail()

In [ ]:
# Aggregate to monthly for easier alignment with quarterly financials
trends_df['month'] = trends_df['date'].dt.to_period('M')
trends_monthly = trends_df.groupby('month').mean(numeric_only=True).reset_index()
trends_monthly['month_dt'] = trends_monthly['month'].dt.to_timestamp()

# Plot
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(trends_monthly['month_dt'], trends_monthly['cava_restaurant'], label='CAVA', linewidth=2, color='#e74c3c')
ax.plot(trends_monthly['month_dt'], trends_monthly['chipotle'], label='Chipotle', linewidth=1.5, linestyle='--', color='#3498db')
ax.plot(trends_monthly['month_dt'], trends_monthly['sweetgreen'], label='Sweetgreen', linewidth=1.5, linestyle=':', color='#2ecc71')
ax.set_title('Google Trends: Search Interest (US, Monthly Avg)', fontweight='bold')
ax.set_ylabel('Search Interest (0-100)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('cava_google_trends.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cava_google_trends.png')

---
## Part 3: Reddit Sentiment (PRAW)

We scrape Reddit posts mentioning CAVA from food/dining/investing subreddits.
Sentiment is scored using VADER (rule-based, no training required).

**Setup required:** Create a Reddit app at https://www.reddit.com/prefs/apps
to get your `client_id` and `client_secret`.

In [ ]:
# !pip install praw vaderSentiment

import praw
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from datetime import datetime

# --- Reddit API credentials ---
# Replace with your own from https://www.reddit.com/prefs/apps
REDDIT_CLIENT_ID = 'YOUR_CLIENT_ID'
REDDIT_CLIENT_SECRET = 'YOUR_CLIENT_SECRET'
REDDIT_USER_AGENT = 'cava_sentiment_scraper/0.1'

reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT
)

analyzer = SentimentIntensityAnalyzer()

# Subreddits to search
subreddits = ['food', 'FoodNYC', 'HealthyFood', 'MediterraneanFood', 
              'investing', 'stocks', 'wallstreetbets']

posts = []
for sub in subreddits:
    try:
        for post in reddit.subreddit(sub).search('CAVA', limit=200, sort='new'):
            sentiment = analyzer.polarity_scores(post.title + ' ' + (post.selftext or ''))
            posts.append({
                'date': datetime.utcfromtimestamp(post.created_utc),
                'subreddit': sub,
                'title': post.title,
                'score': post.score,
                'num_comments': post.num_comments,
                'sentiment_compound': sentiment['compound'],
                'sentiment_pos': sentiment['pos'],
                'sentiment_neg': sentiment['neg'],
                'sentiment_neu': sentiment['neu']
            })
        print(f'r/{sub}: {len([p for p in posts if p["subreddit"]==sub])} posts')
    except Exception as e:
        print(f'r/{sub}: error - {e}')

reddit_df = pd.DataFrame(posts)
reddit_df['date'] = pd.to_datetime(reddit_df['date'])
reddit_df = reddit_df.sort_values('date').reset_index(drop=True)
print(f'\nTotal posts collected: {len(reddit_df)}')
reddit_df.head()

In [ ]:
# Aggregate Reddit sentiment to monthly
reddit_df['month'] = reddit_df['date'].dt.to_period('M')

reddit_monthly = reddit_df.groupby('month').agg(
    post_count=('title', 'count'),
    avg_sentiment=('sentiment_compound', 'mean'),
    avg_upvotes=('score', 'mean'),
    total_comments=('num_comments', 'sum')
).reset_index()
reddit_monthly['month_dt'] = reddit_monthly['month'].dt.to_timestamp()

# Plot sentiment over time
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

# Sentiment
ax = axes[0]
colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in reddit_monthly['avg_sentiment']]
ax.bar(reddit_monthly['month_dt'], reddit_monthly['avg_sentiment'], color=colors, width=25)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Reddit: Monthly Average Sentiment (VADER Compound Score)', fontweight='bold')
ax.set_ylabel('Compound Score (-1 to +1)')

# Volume
ax = axes[1]
ax.bar(reddit_monthly['month_dt'], reddit_monthly['post_count'], color='#3498db', width=25)
ax.set_title('Reddit: Monthly Post Volume Mentioning CAVA', fontweight='bold')
ax.set_ylabel('Number of Posts')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig('cava_reddit_sentiment.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cava_reddit_sentiment.png')

---
## Part 4: Save All Data to CSV

In [ ]:
# Save all datasets
price_df.to_csv('cava_stock_price.csv', index=False)
q_df.to_csv('cava_quarterly_fundamentals.csv', index=False)
trends_monthly.to_csv('cava_google_trends_monthly.csv', index=False)
reddit_monthly.to_csv('cava_reddit_sentiment_monthly.csv', index=False)

print('All data saved:')
print('  cava_stock_price.csv')
print('  cava_quarterly_fundamentals.csv')
print('  cava_google_trends_monthly.csv')
print('  cava_reddit_sentiment_monthly.csv')
print('\nNext step: run cava_modeling.ipynb')